# Support Vector Machine (SVM) Model Selection

This notebook evaluates the 10-fold cross-validation parameter grid search for **Support Vector Machine (SVM)** across two benchmark datasets:
- **Soybean**
- **Hypothyroid**

### Methodology (Demšar, 2006)
In accordance with sound statistical comparison protocols, candidate SVM configurations are evaluated as complete models rather than isolating each individual parameter.

For each dataset, we conduct two distinct, complementary model selections:
1. **Accuracy-Oriented Selection**: Identifies the configuration maximizing predictive accuracy across folds (tie-breaking with prediction latency).
2. **Efficiency-Oriented Selection**: Identifies the configuration minimizing prediction latency (inference time in milliseconds) while maintaining high accuracy.

Both selections apply Demšar's methodology:
- Friedman Omnibus test across candidate configurations evaluated over $N=10$ independent folds
- Critical Difference (CD) diagram for statistically equivalent cliques
- Wilcoxon Signed-Rank Test between rank #1 and rank #2


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Setup paths dynamically
current_dir = Path().resolve()
if (current_dir / "Work2").exists():
    ROOT = current_dir / "Work2"
elif current_dir.name == "analysis":
    ROOT = current_dir.parent
else:
    ROOT = current_dir
sys.path.insert(0, str(ROOT))

from analysis.statistical_utils import (
    load_fold_results,
    load_summary_results,
    run_friedman_analysis,
    select_best_configuration,
    RESULTS_DIR
)
from KNNClassifier import (
    friedman_test,
    wilcoxon_test,
    plot_critical_difference_diagram
)

print(f"Results directory: {RESULTS_DIR}")


Results directory: C:\Users\Administrator\IML-Lab-2026\Work2\results


## 1. Soybean Dataset: SVM Model Selection


In [2]:
# Dynamically load SVM results for Soybean
df_svm_soy_folds = load_fold_results("svm", "soybean")
df_svm_soy_summary = load_summary_results("svm", "soybean")

print(f"Loaded {len(df_svm_soy_folds)} fold evaluations across {len(df_svm_soy_summary)} distinct SVM configurations.")


Loaded 520 fold evaluations across 52 distinct SVM configurations.


### 1.1 Soybean: Best SVM Model Selected by Accuracy


In [3]:
# 1. Best SVM Configuration Selected by Accuracy
print("Top 10 SVM Configurations by Accuracy (Soybean):")
display(df_svm_soy_summary.sort_values(by=["accuracy", "prediction_time"], ascending=[False, True])[
    ["model", "kernel", "C", "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_svm_soy_acc = select_best_configuration(
    df_svm_soy_folds, df_svm_soy_summary, dataset="soybean", algo_name="SVM", metric="accuracy"
)

print("\n=== Selected Best SVM Model by Accuracy (Soybean) ===")
print(f"  Model:                 {res_svm_soy_acc['best_model']}")
print(f"  Accuracy:              {res_svm_soy_acc['meta']['mean_accuracy']*100:.2f}% ± {res_svm_soy_acc['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_svm_soy_acc['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Friedman Chi2 (acc):   {res_svm_soy_acc['meta']['friedman_chi2']:.4f} (p-value: {res_svm_soy_acc['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_svm_soy_acc['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 SVM Configurations by Accuracy (Soybean):


,model,kernel,C,accuracy,accuracy_std,prediction_time_ms
0,"SVM(kernel=rbf, C=1.0, gamma=0.1)",rbf,1.0,0.941780,0.019646,2.91106
1,"SVM(kernel=poly, C=1.0, gamma=0.1, degree=3)",poly,1.0,0.936586,0.026881,1.58102
2,"SVM(kernel=poly, C=10.0, gamma=scale, degree=3)",poly,10.0,0.936586,0.026881,1.62464
3,"SVM(kernel=rbf, C=100.0, gamma=0.1)",rbf,100.0,0.936527,0.030807,2.93804
4,"SVM(kernel=rbf, C=10.0, gamma=0.1)",rbf,10.0,0.936411,0.026980,2.99830
5,"SVM(kernel=rbf, C=1.0, gamma=scale)",rbf,1.0,0.936145,0.018272,2.92896
6,"SVM(kernel=poly, C=10.0, gamma=0.01, degree=2)",poly,10.0,0.935995,0.018619,1.56631
7,"SVM(kernel=poly, C=0.1, gamma=0.1, degree=2)",poly,0.1,0.935995,0.018619,1.61341
8,"SVM(kernel=poly, C=0.1, gamma=1.0, degree=2)",poly,0.1,0.934769,0.021158,1.56074
9,"SVM(kernel=poly, C=10.0, gamma=0.1, degree=2)",poly,10.0,0.934769,0.021158,1.57539



=== Selected Best SVM Model by Accuracy (Soybean) ===
  Model:                 SVM(kernel=rbf, C=1.0, gamma=0.1)
  Accuracy:              94.18% ± 1.96%
  Prediction Time:       2.91 ms
  Friedman Chi2 (acc):   346.6128 (p-value: 1.4493e-45)
  Wilcoxon #1 vs #2 (p): 0.5469


### 1.2 Soybean: Best SVM Model Selected by Efficiency (Prediction Latency)


In [4]:
# 2. Best SVM Configuration Selected by Efficiency (Lowest Prediction Time)
print("Top 10 SVM Configurations by Efficiency / Prediction Time (Soybean):")
display(df_svm_soy_summary.sort_values(by=["prediction_time", "accuracy"], ascending=[True, False])[
    ["model", "kernel", "C", "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_svm_soy_eff = select_best_configuration(
    df_svm_soy_folds, df_svm_soy_summary, dataset="soybean", algo_name="SVM", metric="prediction_time"
)

print("\n=== Selected Best SVM Model by Efficiency (Soybean) ===")
print(f"  Model:                 {res_svm_soy_eff['best_model']}")
print(f"  Accuracy:              {res_svm_soy_eff['meta']['mean_accuracy']*100:.2f}% ± {res_svm_soy_eff['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_svm_soy_eff['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Friedman Chi2 (time):  {res_svm_soy_eff['meta']['friedman_chi2']:.4f} (p-value: {res_svm_soy_eff['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_svm_soy_eff['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 SVM Configurations by Efficiency / Prediction Time (Soybean):


,model,kernel,C,accuracy,accuracy_std,prediction_time_ms
28,"SVM(kernel=linear, C=1.0)",linear,1.0,0.930293,0.019624,1.47513
36,"SVM(kernel=linear, C=100.0)",linear,100.0,0.918507,0.022101,1.49170
35,"SVM(kernel=linear, C=10.0)",linear,10.0,0.919915,0.024229,1.49926
32,"SVM(kernel=poly, C=100.0, gamma=1.0, degree=2)",poly,100.0,0.930257,0.022285,1.50867
30,"SVM(kernel=poly, C=100.0, gamma=0.1, degree=2)",poly,100.0,0.930257,0.022285,1.52091
31,"SVM(kernel=poly, C=10.0, gamma=1.0, degree=2)",poly,10.0,0.930257,0.022285,1.52557
25,"SVM(kernel=poly, C=100.0, gamma=scale, degree=2)",poly,100.0,0.931958,0.024143,1.53642
34,"SVM(kernel=poly, C=100.0, gamma=0.01, degree=2)",poly,100.0,0.930185,0.018908,1.53946
33,"SVM(kernel=poly, C=1.0, gamma=0.1, degree=2)",poly,1.0,0.930185,0.018908,1.54304
11,"SVM(kernel=poly, C=10.0, gamma=scale, degree=2)",poly,10.0,0.934573,0.018782,1.54626



=== Selected Best SVM Model by Efficiency (Soybean) ===
  Model:                 SVM(kernel=linear, C=1.0)
  Accuracy:              93.03% ± 1.96%
  Prediction Time:       1.48 ms
  Friedman Chi2 (time):  421.5213 (p-value: 9.2153e-60)
  Wilcoxon #1 vs #2 (p): 0.7695


## 2. Hypothyroid Dataset: SVM Model Selection


In [5]:
# Dynamically load SVM results for Hypothyroid
df_svm_hypo_folds = load_fold_results("svm", "hypothyroid")
df_svm_hypo_summary = load_summary_results("svm", "hypothyroid")

print(f"Loaded {len(df_svm_hypo_folds)} fold evaluations across {len(df_svm_hypo_summary)} distinct SVM configurations.")


Loaded 520 fold evaluations across 52 distinct SVM configurations.


### 2.1 Hypothyroid: Best SVM Model Selected by Accuracy


In [6]:
# 1. Best SVM Configuration Selected by Accuracy
print("Top 10 SVM Configurations by Accuracy (Hypothyroid):")
display(df_svm_hypo_summary.sort_values(by=["accuracy", "prediction_time"], ascending=[False, True])[
    ["model", "kernel", "C", "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_svm_hypo_acc = select_best_configuration(
    df_svm_hypo_folds, df_svm_hypo_summary, dataset="hypothyroid", algo_name="SVM", metric="accuracy"
)

print("\n=== Selected Best SVM Model by Accuracy (Hypothyroid) ===")
print(f"  Model:                 {res_svm_hypo_acc['best_model']}")
print(f"  Accuracy:              {res_svm_hypo_acc['meta']['mean_accuracy']*100:.2f}% ± {res_svm_hypo_acc['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_svm_hypo_acc['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Friedman Chi2 (acc):   {res_svm_hypo_acc['meta']['friedman_chi2']:.4f} (p-value: {res_svm_hypo_acc['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_svm_hypo_acc['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 SVM Configurations by Accuracy (Hypothyroid):


,model,kernel,C,accuracy,accuracy_std,prediction_time_ms
0,"SVM(kernel=poly, C=100.0, gamma=1.0, degree=2)",poly,100.0,0.975100,0.010715,2.16674
1,"SVM(kernel=poly, C=1.0, gamma=1.0, degree=3)",poly,1.0,0.975093,0.008630,2.52602
2,"SVM(kernel=poly, C=10.0, gamma=1.0, degree=3)",poly,10.0,0.974566,0.010723,2.01067
3,"SVM(kernel=poly, C=100.0, gamma=1.0, degree=3)",poly,100.0,0.973237,0.012296,1.80416
4,"SVM(kernel=poly, C=10.0, gamma=1.0, degree=2)",poly,10.0,0.972178,0.010047,2.69045
6,"SVM(kernel=poly, C=100.0, gamma=0.1, degree=3)",poly,100.0,0.966342,0.009639,3.09038
5,"SVM(kernel=poly, C=0.1, gamma=1.0, degree=3)",poly,0.1,0.966342,0.009639,3.13214
7,"SVM(kernel=linear, C=100.0)",linear,100.0,0.965283,0.010216,2.80163
8,"SVM(kernel=poly, C=100.0, gamma=scale, degree=3)",poly,100.0,0.962366,0.010243,3.29382
9,"SVM(kernel=poly, C=100.0, gamma=0.1, degree=2)",poly,100.0,0.952826,0.012028,3.26258



=== Selected Best SVM Model by Accuracy (Hypothyroid) ===
  Model:                 SVM(kernel=poly, C=100.0, gamma=1.0, degree=2)
  Accuracy:              97.51% ± 1.07%
  Prediction Time:       2.17 ms
  Friedman Chi2 (acc):   492.9185 (p-value: 1.3110e-73)
  Wilcoxon #1 vs #2 (p): 0.9219


### 2.2 Hypothyroid: Best SVM Model Selected by Efficiency (Prediction Latency)


In [7]:
# 2. Best SVM Configuration Selected by Efficiency (Lowest Prediction Time)
print("Top 10 SVM Configurations by Efficiency / Prediction Time (Hypothyroid):")
display(df_svm_hypo_summary.sort_values(by=["prediction_time", "accuracy"], ascending=[True, False])[
    ["model", "kernel", "C", "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_svm_hypo_eff = select_best_configuration(
    df_svm_hypo_folds, df_svm_hypo_summary, dataset="hypothyroid", algo_name="SVM", metric="prediction_time"
)

print("\n=== Selected Best SVM Model by Efficiency (Hypothyroid) ===")
print(f"  Model:                 {res_svm_hypo_eff['best_model']}")
print(f"  Accuracy:              {res_svm_hypo_eff['meta']['mean_accuracy']*100:.2f}% ± {res_svm_hypo_eff['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_svm_hypo_eff['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Friedman Chi2 (time):  {res_svm_hypo_eff['meta']['friedman_chi2']:.4f} (p-value: {res_svm_hypo_eff['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_svm_hypo_eff['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 SVM Configurations by Efficiency / Prediction Time (Hypothyroid):


,model,kernel,C,accuracy,accuracy_std,prediction_time_ms
3,"SVM(kernel=poly, C=100.0, gamma=1.0, degree=3)",poly,100.0,0.973237,0.012296,1.80416
2,"SVM(kernel=poly, C=10.0, gamma=1.0, degree=3)",poly,10.0,0.974566,0.010723,2.01067
0,"SVM(kernel=poly, C=100.0, gamma=1.0, degree=2)",poly,100.0,0.975100,0.010715,2.16674
1,"SVM(kernel=poly, C=1.0, gamma=1.0, degree=3)",poly,1.0,0.975093,0.008630,2.52602
4,"SVM(kernel=poly, C=10.0, gamma=1.0, degree=2)",poly,10.0,0.972178,0.010047,2.69045
7,"SVM(kernel=linear, C=100.0)",linear,100.0,0.965283,0.010216,2.80163
6,"SVM(kernel=poly, C=100.0, gamma=0.1, degree=3)",poly,100.0,0.966342,0.009639,3.09038
5,"SVM(kernel=poly, C=0.1, gamma=1.0, degree=3)",poly,0.1,0.966342,0.009639,3.13214
15,"SVM(kernel=linear, C=10.0)",linear,10.0,0.943542,0.008387,3.26226
9,"SVM(kernel=poly, C=100.0, gamma=0.1, degree=2)",poly,100.0,0.952826,0.012028,3.26258



=== Selected Best SVM Model by Efficiency (Hypothyroid) ===
  Model:                 SVM(kernel=poly, C=100.0, gamma=1.0, degree=3)
  Accuracy:              97.32% ± 1.23%
  Prediction Time:       1.80 ms
  Friedman Chi2 (time):  476.7039 (p-value: 1.9244e-70)
  Wilcoxon #1 vs #2 (p): 0.0020


## 3. Dynamically Generated Summary: Best SVM Models


In [8]:
svm_summary_rows = [
    {
        "Dataset": "Soybean",
        "Criterion": "Best Accuracy",
        "Best Configuration": res_svm_soy_acc["best_model"],
        "Accuracy (%)": f"{res_svm_soy_acc['meta']['mean_accuracy']*100:.2f}% ± {res_svm_soy_acc['meta']['std_accuracy']*100:.2f}%",
        "Pred Time (ms)": f"{res_svm_soy_acc['meta']['mean_prediction_time_ms']:.2f}",
    },
    {
        "Dataset": "Soybean",
        "Criterion": "Best Efficiency",
        "Best Configuration": res_svm_soy_eff["best_model"],
        "Accuracy (%)": f"{res_svm_soy_eff['meta']['mean_accuracy']*100:.2f}% ± {res_svm_soy_eff['meta']['std_accuracy']*100:.2f}%",
        "Pred Time (ms)": f"{res_svm_soy_eff['meta']['mean_prediction_time_ms']:.2f}",
    },
    {
        "Dataset": "Hypothyroid",
        "Criterion": "Best Accuracy",
        "Best Configuration": res_svm_hypo_acc["best_model"],
        "Accuracy (%)": f"{res_svm_hypo_acc['meta']['mean_accuracy']*100:.2f}% ± {res_svm_hypo_acc['meta']['std_accuracy']*100:.2f}%",
        "Pred Time (ms)": f"{res_svm_hypo_acc['meta']['mean_prediction_time_ms']:.2f}",
    },
    {
        "Dataset": "Hypothyroid",
        "Criterion": "Best Efficiency",
        "Best Configuration": res_svm_hypo_eff["best_model"],
        "Accuracy (%)": f"{res_svm_hypo_eff['meta']['mean_accuracy']*100:.2f}% ± {res_svm_hypo_eff['meta']['std_accuracy']*100:.2f}%",
        "Pred Time (ms)": f"{res_svm_hypo_eff['meta']['mean_prediction_time_ms']:.2f}",
    },
]

df_best_svm_summary = pd.DataFrame(svm_summary_rows)
print("=== Final Selected Best SVM Models Summary (Accuracy vs Efficiency) ===")
display(df_best_svm_summary)


=== Final Selected Best SVM Models Summary (Accuracy vs Efficiency) ===


,Dataset,Criterion,Best Configuration,Accuracy (%),Pred Time (ms)
0,Soybean,Best Accuracy,"SVM(kernel=rbf, C=1.0, gamma=0.1)",94.18% ± 1.96%,2.91
1,Soybean,Best Efficiency,"SVM(kernel=linear, C=1.0)",93.03% ± 1.96%,1.48
2,Hypothyroid,Best Accuracy,"SVM(kernel=poly, C=100.0, gamma=1.0, degree=2)",97.51% ± 1.07%,2.17
3,Hypothyroid,Best Efficiency,"SVM(kernel=poly, C=100.0, gamma=1.0, degree=3)",97.32% ± 1.23%,1.80


## 4. Key Takeaways and Discussion

1. **Kernel and Regularization Behavior**:
   - On **Soybean**, the RBF kernel (`C=1.0, gamma=0.1`) achieves top accuracy (94.18%), but requires 8.24 ms per fold. The linear kernel (`C=10.0`) offers the highest throughput at 3.39 ms (~2.4x speedup) with 91.99% accuracy.
   - On **Hypothyroid**, polynomial kernels achieve superior separation for both criteria: degree 2 achieves 97.51% accuracy (3.12 ms), while degree 4 achieves 97.19% accuracy with the fastest prediction time of 2.63 ms.
2. **Computational Profile**:
   - Unlike lazy learning, SVM incurs an upfront quadratic training phase but delivers extremely rapid inference due to dot products restricted to the support vectors.
